In [9]:
import torch
from transformers import AutoModel, AutoTokenizer

# Load model
model_path = "Dream-org/Dream-v0-Instruct-7B"

model = AutoModel.from_pretrained(
    model_path,
    torch_dtype=torch.bfloat16,
    trust_remote_code=True
).cuda().eval()

tokenizer = AutoTokenizer.from_pretrained(
    model_path,
    trust_remote_code=True
)

# ----------- Build infilling prompt -----------

question = "If John has 3 apples and buys 2 more, how many apples does he have?. Povide your reasoning followed by the final answer."
answer = "20"

prefix = f"""{question}
Reasoning:
"""

suffix = f"""
Final Answer: {answer}
"""

# number of mask tokens (important hyperparameter)
num_mask = 128

# Tokenize
prefix_ids = tokenizer.encode(prefix, add_special_tokens=False)
suffix_ids = tokenizer.encode(suffix, add_special_tokens=False)

# Construct input: prefix + masks + suffix
input_ids = (
    [tokenizer.bos_token_id] +
    prefix_ids +
    [tokenizer.mask_token_id] * num_mask +
    suffix_ids +
    [tokenizer.eos_token_id]
)

input_ids = torch.tensor([input_ids]).cuda()

# ----------- Run Dream generation -----------

with torch.no_grad():
    outputs = model.diffusion_generate(
        input_ids,
        max_length=input_ids.shape[1] + 1,
        temperature=0.8,
        top_p=0.95,
        do_sample=True,
        num_inference_steps=64  # diffusion steps (tune this!)
    )

# Decode
decoded = tokenizer.decode(outputs[0], skip_special_tokens=False)

print(decoded)

Loading checkpoint shards: 100%|██████████| 4/4 [00:00<00:00, 199.87it/s]


<|beginoftext|>If John has 3 apples and buys 2 more, how many apples does he have?. Povide your reasoning followed by the final answer.
Reasoning:
- Step 1
- Apple: 3
- Step 1:2 more
- Step 2
- Apple: 3
- Apple: 2
- Step 3
- Add 3 apples and 2 apples
- Final Step
- To find out how many apples John has after buying the new amount, simply add the new amount of apples to to the original amount.
Options:
- A. True
- B. False
- C. 5
- D. 120
- E. 20
- F. 10
- G. 0
Final Answer: 20
<|endoftext|><|endoftext|>
